# Пайплайн: класифікатор → U-Net → постобробка

Етап 8. Усе локально, з ймовірностей, порахованих на Kaggle (`07_inference.ipynb` → `data/inference_v1/`).

Скор одного знімка при порозі класифікатора **c**:

| насправді | класифікатор | скор |
|---|---|---|
| порожньо | p < c | 1 |
| порожньо | p ≥ c | 1, якщо після постобробки не лишилось жодної компоненти, інакше 0 |
| є кораблі | p < c | 0 |
| є кораблі | p ≥ c | F2 масок U-Net |

План: (1) для кожного val-знімка порахувати скор «якщо дійшов до U-Net» для сітки поріг U-Net × min_area;
(2) перебрати c — це вже лише вибір між трьома числами для кожного знімка; (3) перевірити найкращу комбінацію
«повільним» шляхом — `mask_to_rles` + `f2_dataset`; (4) зібрати сабміт для test.

## 1. Дані

In [ ]:
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))
from dataset import rles_by_image
from metric import IOU_THRESHOLDS, f2_dataset
from postprocess import mask_to_rles
from rle import IMAGE_SHAPE, rle_decode

DATA_DIR = ROOT / "data"
INFERENCE_DIR = DATA_DIR / "inference_v1"
OUT_DIR = DATA_DIR / "pipeline_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
CONNECTIVITY = 4

masks = pd.read_csv(DATA_DIR / "train_ship_segmentations_v2.csv")
rles = rles_by_image(masks)

npz = np.load(INFERENCE_DIR / "val_unet_probs.npz", allow_pickle=True)
val_unet = npz["probs"]
val = pd.DataFrame({"ImageId": npz["image_ids"]})
# merge зберігає порядок рядків val, тож i-й рядок відповідає val_unet[i].
val = val.merge(pd.read_csv(DATA_DIR / "classifier_v1" / "val_probs.csv"), on="ImageId", how="left", validate="1:1")
assert val["p"].notna().all()

npz = np.load(INFERENCE_DIR / "test_unet_probs.npz", allow_pickle=True)
test_unet = npz["probs"]
test = pd.DataFrame({"ImageId": npz["image_ids"]})
test = test.merge(pd.read_csv(INFERENCE_DIR / "test_classifier_probs.csv"), on="ImageId", how="left", validate="1:1")
assert test["p"].notna().all()

has_ships = (val["n_ships"] > 0).values
print(f"val: {len(val)} знімків, з кораблями {has_ships.mean():.1%} | test: {len(test)}")
print(f"ймовірності U-Net: val {val_unet.shape}, test {test_unet.shape}, {(val_unet.nbytes + test_unet.nbytes) / 1e9:.1f} ГБ")

### 1.1. Узгодженість з етапом 7

Для знімків з кораблями ймовірності U-Net рахувались і в `06` (`data/postprocess_v1/`). Ваги ті самі, тож
результат має майже збігтися; різниця на 1–2 одиниці з 255 — від AMP (половинна точність залежить від складу батча).

In [ ]:
old = np.load(DATA_DIR / "postprocess_v1" / "val_probs_384.npz", allow_pickle=True)
old_probs = old["probs"]
position = {image_id: i for i, image_id in enumerate(val["ImageId"])}
max_diff, n_big = 0, 0
# По одному знімку: різниця всього масиву в int64 зайняла б ~5 ГБ на кожну копію.
for j, image_id in enumerate(old["image_ids"]):
    diff = np.abs(val_unet[position[image_id]].astype(np.int16) - old_probs[j].astype(np.int16))
    max_diff = max(max_diff, diff.max())
    n_big += (diff > 2).sum()
print(f"знімків: {len(old_probs)} | макс. різниця: {max_diff} з 255 | "
      f"пікселів з різницею > 2: {n_big / old_probs.size:.2e}")
del old, old_probs

## 2. Скор кожного val-знімка, якщо він дійшов до U-Net

Ті самі функції, що в `06`: IoU всіх пар «корабель × компонента» одним `np.bincount`, F2 з матриці IoU.
Для порожнього знімка матриця має 0 рядків: F2 = 1, якщо компонент немає, інакше 0 — окремий випадок не потрібен.

Прискорення: якщо найбільша ймовірність на знімку ≤ найменшого порогу, компонент не буде за жодного порогу
(інтерполяція не дає значень, більших за максимум) — таких знімків більшість.

In [ ]:
def ship_labels(image_rles):
    labels = np.zeros(IMAGE_SHAPE, dtype=np.int32)
    for i, rle in enumerate(image_rles, start=1):
        labels[rle_decode(rle) == 1] = i
    return labels, len(image_rles)


def ious_from_labels(true_labels, n_true, pred_labels, n_pred):
    pairs = true_labels.astype(np.int64) * (n_pred + 1) + pred_labels
    joint = np.bincount(pairs.ravel(), minlength=(n_true + 1) * (n_pred + 1)).reshape(n_true + 1, n_pred + 1)
    intersection = joint[1:, 1:]
    true_areas = joint[1:, :].sum(axis=1)
    pred_areas = joint[:, 1:].sum(axis=0)
    return intersection / (true_areas[:, None] + pred_areas[None, :] - intersection)


def f2_from_ious(ious):
    n_true, n_pred = ious.shape
    if n_true == 0 and n_pred == 0:
        return 1.0
    scores = []
    for threshold in IOU_THRESHOLDS:
        tp = (ious > threshold).any(axis=1).sum()
        scores.append(5 * tp / (5 * tp + 4 * (n_true - tp) + (n_pred - tp)))
    return np.mean(scores)


def upscale(probs_384):
    return cv2.resize(probs_384.astype(np.float32) / 255, IMAGE_SHAPE[::-1], interpolation=cv2.INTER_LINEAR)

In [ ]:
UNET_THRESHOLDS = [0.5, 0.6, 0.7, 0.8]
MIN_AREAS = [0, 20, 40, 80, 150, 300]

start = time.time()
unet_scores = np.zeros((len(val), len(UNET_THRESHOLDS), len(MIN_AREAS)), dtype=np.float32)
for i, image_id in enumerate(val["ImageId"]):
    image_rles = rles.get(image_id, [])
    if val_unet[i].max() / 255 <= UNET_THRESHOLDS[0]:
        unet_scores[i] = 0.0 if image_rles else 1.0
        continue
    true_labels, n_true = ship_labels(image_rles)
    probs_768 = upscale(val_unet[i])
    for t, threshold in enumerate(UNET_THRESHOLDS):
        mask = (probs_768 > threshold).astype(np.uint8)
        n_labels, pred_labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=CONNECTIVITY)
        ious = ious_from_labels(true_labels, n_true, pred_labels, n_labels - 1)
        areas = stats[1:, cv2.CC_STAT_AREA]
        for a, min_area in enumerate(MIN_AREAS):
            unet_scores[i, t, a] = f2_from_ious(ious[:, areas >= min_area])
    if (i + 1) % 5000 == 0:
        print(f"{i + 1} знімків, {time.time() - start:.0f} с")
print(f"готово: {time.time() - start:.0f} с")

## 3. Перебір порогу класифікатора

Для кожного c скор знімка — один з трьох варіантів (таблиця вгорі), середнє по всій val. Для порівняння:
- **без класифікатора** (c = 0) — U-Net на кожному знімку;
- **ідеальний класифікатор** — U-Net рівно на знімках з кораблями; це стеля для нашої U-Net і постобробки.

In [ ]:
CLASSIFIER_THRESHOLDS = np.round(np.arange(0.05, 1.0, 0.05), 2)
skipped_score = np.where(has_ships, 0.0, 1.0)[:, None, None]


def pipeline_f2(passed):
    return np.where(passed[:, None, None], unet_scores, skipped_score).mean(axis=0)


rows = []
for c in CLASSIFIER_THRESHOLDS:
    f2 = pipeline_f2(val["p"].values >= c)
    for t, threshold in enumerate(UNET_THRESHOLDS):
        for a, min_area in enumerate(MIN_AREAS):
            rows.append((c, threshold, min_area, f2[t, a]))
grid = pd.DataFrame(rows, columns=["classifier_threshold", "unet_threshold", "min_area", "f2"])
grid.to_csv(OUT_DIR / "val_pipeline_grid.csv", index=False)

best = grid.loc[grid["f2"].idxmax()]
t_best = UNET_THRESHOLDS.index(best["unet_threshold"])
a_best = MIN_AREAS.index(best["min_area"])
no_classifier = pipeline_f2(np.ones(len(val), dtype=bool))[t_best, a_best]
ideal_classifier = pipeline_f2(has_ships)[t_best, a_best]

print(f"найкраще: c = {best['classifier_threshold']}, поріг U-Net {best['unet_threshold']}, "
      f"min_area {int(best['min_area'])} → val F2 {best['f2']:.4f}")
print(f"тривіальний сабміт:      0.7790")
print(f"без класифікатора:       {no_classifier:.4f}")
print(f"ідеальний класифікатор:  {ideal_classifier:.4f}")

In [ ]:
for threshold in UNET_THRESHOLDS:
    table = grid[grid["unet_threshold"] == threshold].pivot(index="classifier_threshold", columns="min_area", values="f2")
    print(f"\nпоріг U-Net {threshold} (рядки — поріг класифікатора c, стовпці — min_area):")
    print(table.iloc[::2].round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for min_area in [0, 40, 150, 300]:
    curve = grid[(grid["unet_threshold"] == best["unet_threshold"]) & (grid["min_area"] == min_area)]
    ax.plot(curve["classifier_threshold"], curve["f2"], marker="o", markersize=4, label=f"min_area {min_area}")
ax.axhline(0.7790, color="gray", linestyle="--", linewidth=1, label="тривіальний сабміт")
ax.axhline(ideal_classifier, color="black", linestyle=":", linewidth=1, label="ідеальний класифікатор")
ax.set_xlabel("поріг класифікатора c")
ax.set_ylabel("val F2 (вся val)")
ax.set_title(f"F2 пайплайну, поріг U-Net {best['unet_threshold']}")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "val_f2_vs_threshold.png", dpi=80)
plt.show()

### 3.1. Де втрачається скор

Кожен знімок дає від 0 до 1, тож 1 − F2 розкладається на частки втрат (у частках від усієї val):
- **порожні знімки, де лишилась компонента** — FP: кожен коштує цілу 1;
- **знімки з кораблями, відсіяні класифікатором** — FN класифікатора: теж ціла 1;
- **знімки з кораблями, що дійшли до U-Net** — недобрані бали F2 (пропущені кораблі, низький IoU, зайві компоненти).

In [ ]:
passed = val["p"].values >= best["classifier_threshold"]
final = np.where(passed, unet_scores[:, t_best, a_best], skipped_score[:, 0, 0])
n = len(val)
losses = {
    "порожні знімки з компонентою (FP)": ((~has_ships) & (final == 0)).sum() / n,
    "кораблі, відсіяні класифікатором": (has_ships & ~passed).sum() / n,
    "кораблі, що дійшли до U-Net": (1 - final[has_ships & passed]).sum() / n,
}
for name, loss in losses.items():
    print(f"{name:40s} −{loss:.4f}")
print(f"{'разом':40s} −{sum(losses.values()):.4f}  (1 − F2 = {1 - final.mean():.4f})")
print(f"\nпорожніх знімків з компонентою: {((~has_ships) & (final == 0)).sum()} з {(~has_ships).sum()}; "
      f"відсіяно знімків з кораблями: {(has_ships & ~passed).sum()} з {has_ships.sum()}")

## 4. Перевірка «повільним» шляхом

Швидкий підрахунок вище — власний код. Перевіряємо його справжнім шляхом сабміту: `mask_to_rles` → таблиця
формату сабміту → `f2_dataset` з `src/metric.py` на 1 000 випадкових val-знімках. Скор має збігтися.

In [ ]:
def predict_rows(image_ids, classifier_p, unet_probs):
    rows = []
    for image_id, p, probs in zip(image_ids, classifier_p, unet_probs):
        image_rles = []
        if p >= best["classifier_threshold"]:
            image_rles = mask_to_rles(probs / 255, threshold=best["unet_threshold"],
                                      min_area=best["min_area"], connectivity=CONNECTIVITY)
        rows += [(image_id, rle) for rle in image_rles] or [(image_id, None)]
    return pd.DataFrame(rows, columns=["ImageId", "EncodedPixels"])


sample = np.random.default_rng(SEED).choice(len(val), size=min(1000, len(val)), replace=False)
sample_ids = val["ImageId"].values[sample]
predicted = predict_rows(sample_ids, val["p"].values[sample], val_unet[sample])
slow = f2_dataset(masks[masks["ImageId"].isin(sample_ids)], predicted)
fast = final[sample].mean()
print(f"1000 знімків: швидкий шлях {fast:.4f}, повільний {slow:.4f}")
assert np.isclose(fast, slow)

## 5. Сабміт

Ті самі параметри — на test, без жодних змін. Перевірки формату: кожен test-знімок присутній, порожній знімок —
рівно один рядок з порожнім `EncodedPixels`, маски одного знімка не перекриваються (за побудовою: компоненти
не перетинаються, але перевіряємо файл, який реально піде на Kaggle).

In [ ]:
start = time.time()
submission = predict_rows(test["ImageId"], test["p"], test_unet)
submission.to_csv(DATA_DIR / "submission_v1.csv", index=False)
print(f"рядків: {len(submission)}, {time.time() - start:.0f} с")

expected = set(pd.read_csv(DATA_DIR / "sample_submission_v2.csv")["ImageId"])
assert set(submission["ImageId"]) == expected
ships = submission.dropna(subset=["EncodedPixels"])
assert not submission[submission["EncodedPixels"].isna()]["ImageId"].isin(ships["ImageId"]).any()
for image_id, group in ships.groupby("ImageId"):
    total = sum(rle_decode(rle) for rle in group["EncodedPixels"])
    assert total.max() <= 1, image_id
print(f"знімків з кораблями: {ships['ImageId'].nunique()} з {len(expected)} "
      f"({ships['ImageId'].nunique() / len(expected):.1%}), кораблів: {len(ships)}")
print("формат: OK →", DATA_DIR / "submission_v1.csv")